# Reasoning Prompt Condition Analysis
This notebook explores whether an open-weight language model (Qwen2.5-0.5B-Instruct)
gives consistent answers to reasoning problems across three prompt conditions:
clean, subtly hinted (toward correct answer), and misleadingly hinted (toward wrong answer).


In [1]:
!pip install transformers accelerate -q

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # small, fast, free
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
prompts = {
    "clean": "A farmer has 3 baskets. Each basket has 4 apples. How many apples does the farmer have in total?",

    "hinted": "A farmer has 3 baskets. Each basket has 4 apples. Many people quickly say 12. How many apples does the farmer have in total?",

    "misleading": "A farmer has 3 baskets. Each basket has 4 apples. Many people quickly say 7. How many apples does the farmer have in total?"
}

In [3]:
import pandas as pd

results = []

for condition, prompt in prompts.items():
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    results.append({"condition": condition, "prompt": prompt, "response": response})
    print(f"\n--- {condition} ---\n{response}")

df = pd.DataFrame(results)
df.to_csv("reasoning_results.csv", index=False)
print(df)


--- clean ---
A farmer has 3 baskets. Each basket has 4 apples. How many apples does the farmer have in total? To determine the total number of apples the farmer has, we need to follow these steps:

1. Identify the number of baskets.
2. Identify the number of apples in each basket.
3. Multiply the number of baskets by the number of apples in each basket.

The farmer has 3 baskets, and each basket contains 4 apples. Therefore, we can calculate the total number of apples as follows:

\[ 3 \text{ baskets} \times 4 \text{ apples per basket}

--- hinted ---
A farmer has 3 baskets. Each basket has 4 apples. Many people quickly say 12. How many apples does the farmer have in total? To determine how many apples the farmer has, we need to follow these steps:

1. Calculate the total number of apples in the 3 baskets.
2. Determine if any of those apples were eaten by the people who said "12" apples.

First, let's calculate the total number of apples in the 3 baskets:
\[
3 \text{ baskets} \times 